# 7. Evaluation and benchmarking

Report exact match, character error rate and normalized Levenshtein similarity together. For prices retain decimals/currency in raw exact match and report the separately defined whole-string numeric metric. Keep dataset counts, image policy, batch size, precision and hardware beside every score.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json
ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "waterec_ocr").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook inside the repository")
sys.path.insert(0, str(ROOT))
from waterec_ocr.waterec_config import load_config
from waterec_ocr.waterec_models import default_checkpoint
CONFIG_PATH = os.environ.get("WATEREC_CONFIG", "configs/dummy_alpr.yaml")
config = load_config(CONFIG_PATH)
checkpoint = default_checkpoint(config)
deployment = config["output_dir"] / "deployment"
examples = sorted((config["dataset_root"] / "val").glob("*.png"))
IMAGE_PATH = examples[0] if examples else config["dataset_root"] / "val/example.png"
def run(*args):
    return subprocess.run([sys.executable, "-m", "waterec_ocr.waterec_cli", *args, "--config", CONFIG_PATH], cwd=ROOT, check=True)
print("Profile:", CONFIG_PATH, "variant:", config["variant"], "patch budget:", config["tokens"])


In [ ]:
RUN_EVALUATION = checkpoint.is_file()
if RUN_EVALUATION:
    run("evaluate", "--backend", "torch", "--folder", str(config["dataset_root"] / "val"), "--output", str(config["output_dir"] / "evaluation_native"))


In [ ]:
RUN_BENCHMARK = os.environ.get("RUN_BENCHMARK", "0") == "1"
if RUN_BENCHMARK:
    for rgb in (False, True):
        scope = "rgb" if rgb else "file_to_text"
        extra = ["--predecoded"] if rgb else []
        run("benchmark", "--backend", "tensorrt", "--folder", str(config["dataset_root"] / "val"), "--repeats", "3", "--output", str(config["output_dir"] / f"benchmark_{scope}.json"), *extra)


RGB timing excludes file decoding; file-to-text includes it. Both exclude network transfer, detection, model loading and compilation. Use fresh processes and warmups. NVML GPU memory includes CUDA context; host RAM and checkpoint size are separate. Four dummy validation images are a functional fixture, not a throughput dataset. Read `results/reference_metrics.json` for the actual experimental cohorts and resource scopes.